In [1]:
import yfinance as yf #yahoo finance
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
stock_data=yf.download('AAPL',start='2025-01-01') #'AAPL' is the ticker for the apple data and we should know the ticker to import any company data
stock_data.head(10)

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2025-01-02,242.525177,247.746654,240.506207,247.577564,55740700
2025-01-03,242.037827,242.853364,240.575812,242.037827,40244100
2025-01-06,243.668915,245.986258,241.878691,242.982661,45045600
2025-01-07,240.894089,244.215939,240.038760,241.659894,40856000
2025-01-08,241.381409,242.385931,238.745812,240.605648,37628900
2025-01-10,235.563202,238.855216,231.734113,238.706022,61710900
2025-01-13,233.126511,233.395048,228.471944,232.261242,49630700
2025-01-14,232.012573,234.837140,231.206976,233.474588,39435300


In [2]:
data=stock_data['Close'].dropna()
data.head(10)

Ticker,AAPL
Date,
2025-01-02,242.525177
2025-01-03,242.037827
2025-01-06,243.668915
2025-01-07,240.894089
2025-01-08,241.381409
2025-01-10,235.563202
2025-01-13,233.126511
2025-01-14,232.012573
2025-01-15,236.577637


In [3]:
import pandas as pd
import numpy as np

from statsmodels.tsa.stattools import adfuller, kpss
from scipy.stats import kstest

##Augmented Dickey-Fuller Test (ADF)

In [4]:
from statsmodels.tsa.stattools import adfuller

result = adfuller(stock_data['Close'].dropna())

print("ADF Statistic:", result[0])
print("p-value:", result[1])

print("Critical Values:")
for key, value in result[4].items():
    print(key, ":", value)

ADF Statistic: -1.4126412415872263
p-value: 0.5762009686550527
Critical Values:
1% : -3.452636878592149
5% : -2.8713543954331433
10% : -2.5719993576515705


##KPSS Test

In [6]:
from statsmodels.tsa.stattools import kpss
result=kpss(stock_data['Close'].dropna(),regression='ct') #ct trend stationarity
print("KPSS Statistic:", result[0])
print("p-value:", result[1])

print("Critical Values:")
for key, value in result[3].items():
    print(key, ":", value)

KPSS Statistic: 0.43540523567970607
p-value: 0.01
Critical Values:
10% : 0.119
5% : 0.146
2.5% : 0.176
1% : 0.216


/tmp/ipykernel_368/2178496840.py:2: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  result=kpss(stock_data['Close'].dropna(),regression='ct') #ct trend stationarity


##Kolmogorov–Smirnov Test (KS Test)
#It checks whether the data follows a specific distribution (usually normal distribution).

In [7]:
from scipy.stats import kstest

data = stock_data['Close'].dropna()

ks_stat, p_value = kstest(data, 'norm')

print("KS Statistic:", ks_stat)
print("p-value:", p_value)

KS Statistic: [1.]
p-value: [0.]


##transformation techniques to make data stationary

##Differencing (Most Common Technique)

In [22]:
stock_data['first_difference'] = stock_data[('Close','AAPL')].diff()

print(stock_data[['Close','first_difference']])

Price            Close first_difference
Ticker            AAPL                 
Date                                   
2025-01-02  242.525177              NaN
2025-01-03  242.037827        -0.487350
2025-01-06  243.668915         1.631088
2025-01-07  240.894089        -2.774826
2025-01-08  241.381409         0.487320
...                ...              ...
2026-03-06  257.459991        -2.830017
2026-03-09  259.880005         2.420013
2026-03-10  260.829987         0.949982
2026-03-11  260.809998        -0.019989
2026-03-12  255.759995        -5.050003

[298 rows x 2 columns]


##logarithmic transformation

In [25]:
import numpy as np

stock_data['log_close'] = np.log(stock_data['Close','AAPL'])
stock_data['log_close']

,log_close
Date,
2025-01-02,5.491106
2025-01-03,5.489094
2025-01-06,5.495810
2025-01-07,5.484357
2025-01-08,5.486378
...,...
2026-03-06,5.550864
2026-03-09,5.560220
2026-03-10,5.563869


##power

In [27]:
import numpy as np

stock_data['sqrt_close'] = np.sqrt(stock_data['Close','AAPL'])
stock_data['sqrt_close']

,sqrt_close
Date,
2025-01-02,15.573220
2025-01-03,15.557565
2025-01-06,15.609898
2025-01-07,15.520763
2025-01-08,15.536454
...,...
2026-03-06,16.045560
2026-03-09,16.120794
2026-03-10,16.150232


##Moving Average Smoothing

In [31]:
stock_data['moving_avg'] = stock_data['Close','AAPL'].rolling(window=12).mean()
stock_data['moving_avg']

,moving_avg
Date,
2025-01-02,NaN
2025-01-03,NaN
2025-01-06,NaN
2025-01-07,NaN
2025-01-08,NaN
...,...
2026-03-06,265.298332
2026-03-09,265.240000
2026-03-10,264.927500


##Seasonal Differencing

In [32]:
stock_data['seasonal_diff'] = stock_data['Close','AAPL'] - stock_data['Close','AAPL'].shift(12)
stock_data['seasonal_diff']

,seasonal_diff
Date,
2025-01-02,NaN
2025-01-03,NaN
2025-01-06,NaN
2025-01-07,NaN
2025-01-08,NaN
...,...
2026-03-06,-6.890015
2026-03-09,-0.699982
2026-03-10,-3.750000


##detrending

In [36]:
from statsmodels.tsa.seasonal import seasonal_decompose

decomposition = seasonal_decompose(stock_data['Close','AAPL'], model='additive', period=30)

stock_data['detrended'] = stock_data['Close','AAPL'] - decomposition.trend
stock_data['detrended']

,detrended
Date,
2025-01-02,NaN
2025-01-03,NaN
2025-01-06,NaN
2025-01-07,NaN
2025-01-08,NaN
...,...
2026-03-06,NaN
2026-03-09,NaN
2026-03-10,NaN


##box-cox transformation

In [41]:
#box cox require all positive values
from scipy.stats import boxcox

stock_data['boxcox'], lambda_value = boxcox(stock_data['Close','AAPL'])
stock_data['boxcox']

,boxcox
Date,
2025-01-02,180.755953
2025-01-03,180.414169
2025-01-06,181.557894
2025-01-07,179.611879
2025-01-08,179.953746
...,...
2026-03-06,191.208814
2026-03-09,192.898831
2026-03-10,193.561972


##finally checking the stationarity of the data after making the data statioanary by using ADF technique

In [42]:
from statsmodels.tsa.stattools import adfuller

result = adfuller(stock_data['first_difference'].dropna())

print("ADF Statistic:", result[0])
print("p-value:", result[1])

for key, value in result[4].items():
    print("Critical Value:", key, value)

ADF Statistic: -15.723069656029494
p-value: 1.3167540093955162e-28
Critical Value: 1% -3.452636878592149
Critical Value: 5% -2.8713543954331433
Critical Value: 10% -2.5719993576515705
